## STEP 3

### Labs Extraction Pipeline (D_LABITEMS Mapping → 24h Aggregation)

In [1]:
import os, json
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/mimic_data"
OUT_DIR  = "/content/drive/MyDrive/mimic_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

D_LABITEMS_PATH = os.path.join(DATA_DIR, "D_LABITEMS.csv")

LAB_PATTERNS = {
    "lactate": ["lactate"],
    "creatinine": ["creatinine"],
    "bun": ["urea nitrogen", "bun"],
    "wbc": ["wbc", "white blood cell"],
    "hemoglobin": ["hemoglobin"],
    "platelets": ["platelet"],
    "sodium": ["sodium"],
    "potassium": ["potassium"],
    "chloride": ["chloride"],
    "bicarbonate": ["bicarbonate"],
    "bilirubin_total": ["bilirubin, total"],
    "inr": ["inr", "pt(inr)"],
}

d = pd.read_csv(D_LABITEMS_PATH, usecols=["ITEMID","LABEL"])
d.columns = d.columns.str.lower()
d["label_l"] = d["label"].astype(str).str.lower()

mapping = {}
for concept, pats in LAB_PATTERNS.items():
    mask = False
    for p in pats:
        mask = mask | d["label_l"].str.contains(p, na=False)
    itemids = sorted(d.loc[mask, "itemid"].dropna().astype(int).unique().tolist())
    mapping[concept] = itemids

out_json = os.path.join(OUT_DIR, "lab_itemids.json")
with open(out_json, "w") as f:
    json.dump(mapping, f, indent=2)

print("✅ Saved:", out_json)
for k, v in mapping.items():
    print(k, "n=", len(v), "sample=", v[:10])

# audit
audit = []
for concept, itemids in mapping.items():
    sub = d[d["itemid"].isin(itemids)][["itemid","label"]].drop_duplicates()
    sub["concept"] = concept
    audit.append(sub)
audit_df = pd.concat(audit).sort_values(["concept","itemid"])
audit_csv = os.path.join(OUT_DIR, "lab_itemid_audit.csv")
audit_df.to_csv(audit_csv, index=False)
print("✅ Saved audit:", audit_csv)

/tmp/ipython-input-148/438416127.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = mask | d["label_l"].str.contains(p, na=False)


✅ Saved: /content/drive/MyDrive/mimic_outputs/lab_itemids.json
lactate n= 5 sample= [50813, 50843, 50954, 51015, 51054]
creatinine n= 13 sample= [50841, 50912, 51021, 51032, 51052, 51067, 51070, 51073, 51080, 51081]
bun n= 4 sample= [50851, 51006, 51045, 51104]
wbc n= 11 sample= [51128, 51300, 51301, 51363, 51384, 51439, 51458, 51516, 51517, 51518]
hemoglobin n= 11 sample= [50805, 50811, 50814, 50852, 50855, 51212, 51222, 51223, 51224, 51225]
platelets n= 4 sample= [51240, 51264, 51265, 51266]
sodium n= 8 sample= [50824, 50834, 50848, 50983, 51042, 51058, 51065, 51100]
potassium n= 8 sample= [50822, 50833, 50847, 50971, 51041, 51057, 51064, 51097]
chloride n= 8 sample= [50806, 50839, 50902, 51013, 51030, 51050, 51062, 51078]
bicarbonate n= 7 sample= [50803, 50837, 50882, 51027, 51048, 51061, 51076]
bilirubin_total n= 5 sample= [50838, 50885, 51012, 51028, 51049]
inr n= 1 sample= [51237]
✅ Saved audit: /content/drive/MyDrive/mimic_outputs/lab_itemid_audit.csv


In [2]:
path = os.path.join(OUT_DIR, "lab_itemids.json")

with open(path, "r") as f:
    m = json.load(f)

m["lactate"] = [50813]
m["creatinine"] = [50912, 50841]
m["bun"] = [50851]
m["wbc"] = [51300, 51301]
m["hemoglobin"] = [51222, 50811]
m["platelets"] = [51265, 51266]
m["sodium"] = [50983]
m["potassium"] = [50971]
m["chloride"] = [50902]
m["bicarbonate"] = [50882]
m["bilirubin_total"] = [50885]
m["inr"] = [51237]

with open(path, "w") as f:
    json.dump(m, f, indent=2)

print("updated lab_itemids.json saved.")

updated lab_itemids.json saved.


In [3]:
import numpy as np

COHORT_PATH = os.path.join(OUT_DIR, "cohort_icustay.parquet")
LAB_ITEMIDS_PATH = os.path.join(OUT_DIR, "lab_itemids.json")
LAB_SUBSET_PATH = os.path.join(DATA_DIR, "labevents_labs_subset.parquet")

# 1) Cohort: only need hadm_id / icustay_id / intime (ICU admission anchor)
cohort = pd.read_parquet(COHORT_PATH)[["hadm_id", "icustay_id", "intime"]].copy()
cohort["intime"] = pd.to_datetime(cohort["intime"], errors="coerce")
cohort["hadm_id"] = cohort["hadm_id"].astype(int)
cohort["icustay_id"] = cohort["icustay_id"].astype(int)

# 2) Lab mapping: ITEMID -> lab concept (from curated lab_itemids.json)
with open(LAB_ITEMIDS_PATH, "r") as f:
    lab_map = json.load(f)
itemid_to_concept = {int(iid): concept for concept, ids in lab_map.items() for iid in ids}

# 3) Load LABEVENTS subset (already filtered by ITEMIDs on the local machine)
labs = pd.read_parquet(LAB_SUBSET_PATH)
labs["charttime"] = pd.to_datetime(labs["charttime"], errors="coerce")
labs = labs.dropna(subset=["hadm_id", "charttime", "itemid", "valuenum"])
labs["hadm_id"] = labs["hadm_id"].astype(int)
labs["itemid"] = labs["itemid"].astype(int)
labs["concept"] = labs["itemid"].map(itemid_to_concept)

# Keep only rows whose ITEMID is in our mapping (safety check)
labs = labs[labs["concept"].notna()].copy()

print("labs rows:", len(labs), "unique hadm:", labs["hadm_id"].nunique())

# 4) Assign each lab event to an ICU stay (icustay_id) within the same hospitalization (HADM_ID)
# Output schema: icustay_id, concept, charttime, valuenum
assigned_parts = []

# Pre-group cohort by hadm_id to get ICU admission windows (intime -> intime+24h)
cohort_by_hadm = {
    hid: g[["icustay_id", "intime"]].sort_values("intime")
    for hid, g in cohort.groupby("hadm_id")
}

for hid, g_lab in labs.groupby("hadm_id"):
    if hid not in cohort_by_hadm:
        continue

    win = cohort_by_hadm[hid]

    # ICU windows for this admission
    icu_ids = win["icustay_id"].to_numpy()
    starts = win["intime"].to_numpy(dtype="datetime64[ns]")
    ends = (win["intime"] + pd.Timedelta(hours=24)).to_numpy(dtype="datetime64[ns]")

    t = g_lab["charttime"].to_numpy(dtype="datetime64[ns]")

    # For each lab event, check which ICU 24h window it falls into: (t>=start) & (t<end)
    # The boolean matrix is feasible because the number of ICU stays per HADM_ID is usually small (1–3).
    mask = (t[:, None] >= starts[None, :]) & (t[:, None] < ends[None, :])

    # If an event matches multiple windows (rare), assign it to the closest ICU start time
    diff = (t[:, None] - starts[None, :]).astype("timedelta64[ns]").astype("int64")
    diff[~mask] = np.iinfo(np.int64).max

    best_idx = diff.argmin(axis=1)
    has_match = mask.any(axis=1)

    out = g_lab.loc[has_match, ["concept", "charttime", "valuenum"]].copy()
    out["icustay_id"] = icu_ids[best_idx[has_match]]
    assigned_parts.append(out[["icustay_id", "concept", "charttime", "valuenum"]])

assigned = (
    pd.concat(assigned_parts, ignore_index=True)
    if assigned_parts
    else pd.DataFrame(columns=["icustay_id", "concept", "charttime", "valuenum"])
)
print("assigned rows:", len(assigned), "unique icustay:", assigned["icustay_id"].nunique())

# 5) Aggregate to icustay-level features (mean/min/max/std/median/count/last)
assigned = assigned.sort_values(["icustay_id", "concept", "charttime"])
g = assigned.groupby(["icustay_id", "concept"])["valuenum"]
agg = g.agg(["mean", "min", "max", "std", "median", "count"]).reset_index()

# "last" is the most recent value within the 24h window
last = (
    assigned.groupby(["icustay_id", "concept"]).tail(1)
    .loc[:, ["icustay_id", "concept", "valuenum"]]
    .rename(columns={"valuenum": "last"})
)
agg = agg.merge(last, on=["icustay_id", "concept"], how="left")

wide = agg.pivot(index="icustay_id", columns="concept")
wide.columns = [f"{concept}_{stat}" for stat, concept in wide.columns]
wide = wide.reset_index()

labs_out_path = os.path.join(OUT_DIR, "labs_24h.parquet")
wide.to_parquet(labs_out_path, index=False, compression="snappy")
print("✅ saved:", labs_out_path, "shape:", wide.shape)

labs rows: 5752022 unique hadm: 58094
assigned rows: 980701 unique icustay: 57670
✅ saved: /content/drive/MyDrive/mimic_outputs/labs_24h.parquet shape: (57670, 78)


In [4]:
OUT_DIR  = "/content/drive/MyDrive/mimic_outputs"
train_path = os.path.join(OUT_DIR, "train_structured_vitals_24h.parquet")
labs_path  = os.path.join(OUT_DIR, "labs_24h.parquet")

train = pd.read_parquet(train_path)
labs  = pd.read_parquet(labs_path)

train2 = train.merge(labs, on="icustay_id", how="left")
out_path = os.path.join(OUT_DIR, "train_structured_vitals_labs_24h.parquet")
train2.to_parquet(out_path, index=False, compression="snappy")

print("✅ saved:", out_path, "shape:", train2.shape)

✅ saved: /content/drive/MyDrive/mimic_outputs/train_structured_vitals_labs_24h.parquet shape: (61532, 128)


In [5]:
TRAIN_PATH = "/content/drive/MyDrive/mimic_outputs/train_structured_vitals_labs_24h.parquet"
train2 = pd.read_parquet(TRAIN_PATH)

# Feature groups
vital_cols = [c for c in train2.columns if c.startswith(("heart_rate_", "resp_rate_", "spo2_", "map_", "temperature_c_"))]
lab_prefixes = ["lactate", "creatinine", "bun", "wbc", "hemoglobin", "platelets",
                "sodium", "potassium", "chloride", "bicarbonate", "bilirubin_total", "inr"]
lab_cols = [c for c in train2.columns if any(c.startswith(x + "_") for x in lab_prefixes)]

# Sanity checks
all_labs_missing = train2[lab_cols].isna().all(axis=1).mean()
all_vitals_missing = train2[vital_cols].isna().all(axis=1).mean()
missing_both = (train2[vital_cols].isna().all(axis=1) & train2[lab_cols].isna().all(axis=1)).mean()

print(f"rows: {len(train2):,}")
print(f"fraction with all vitals missing: {all_vitals_missing:.4f}")
print(f"fraction with all labs missing:   {all_labs_missing:.4f}")
print(f"fraction missing both:            {missing_both:.4f}")
print(f"#vital features: {len(vital_cols)}, #lab features: {len(lab_cols)}")

rows: 61,532
fraction with all vitals missing: 0.0399
fraction with all labs missing:   0.0628
fraction missing both:            0.0203
#vital features: 35, #lab features: 77
